In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import uproot
import sys
import math
import ROOT




B = 15. 

L = 48.

h = 180.

def m2(theta):
    return np.tan(theta)

m1 = h/B

b1 = -1*m1*(B-L)

b2 = h

def coverage(theta):
    xint = (b2 - b1) / (m1 - m2(theta))
    return (L - B) + xint



h = ROOT.TH1D("h", "", 16, 0, B)

angles = np.linspace(0, math.pi/2, 100)

for angle in angles:
    c = coverage(angle)
    for num in range(1, h.GetNbinsX()+1):
        if (h.GetBinCenter(num) - h.GetBinWidth(num)/2.0) > (B - c):
            h.Fill(h.GetBinCenter(num))

c = ROOT.TCanvas("c", "c", 700, 500)
c.SetLeftMargin(0.15)
h.SetStats(0)
h.GetXaxis().SetTitle("Fake Reconstructed Z [cm]")
h.GetYaxis().SetTitle("Fake Track Count")
h.Draw("HIST")

c.Draw()

In [ ]:

h_pdf = ROOT.TH1D("h_pdf", "", 1000, 0, math.pi/2)

for num in range(1, h_pdf.GetNbinsX()+1):
    a = h_pdf.GetBinCenter(num)
    v = np.cos(a)**2
    v *= np.sin(a)
    h_pdf.SetBinContent(num, v)

h_pdf.Scale(1.0/h_pdf.Integral())

c = ROOT.TCanvas("c", "c", 700, 500)
c.SetLeftMargin(0.15)
h_pdf.SetStats(0)
h_pdf.GetXaxis().SetTitle("Angle [radians]")
h_pdf.GetYaxis().SetTitle("Prob. Density")
h_pdf.Draw("HIST")
c.Draw()

In [ ]:
import random

print(random.random())

def sample_angle(pdf):
    for num in range(10000000):
        x = random.random()*(math.pi/2)
        y = random.random()*pdf.GetMaximum()
        if y < pdf.GetBinContent(pdf.FindBin(x)):
            return x
        
    return 0

h_test = ROOT.TH1D("h_test", "", 1000, 0, math.pi/2)
for num in range(100000):
    h_test.Fill(sample_angle(h_pdf))
    
c = ROOT.TCanvas("c", "c", 700, 500)
c.SetLeftMargin(0.15)
h_test.SetStats(0)
h_test.GetXaxis().SetTitle("Angle [radians]")
h_test.GetYaxis().SetTitle("Counts")
h_test.Draw("HIST")
c.Draw()

In [ ]:
def populate(h, pdf, N):
    for num in range(N):
        angle = (math.pi/2) - sample_angle(h_pdf)
        c = coverage(angle)
        for num in range(1, h.GetNbinsX()+1):
            if (h.GetBinCenter(num) - h.GetBinWidth(num)/2.0) > (B - c):
                h.Fill(h.GetBinCenter(num))

h_final = ROOT.TH1D("h_final", "", 16, 0, B)
populate(h_final, h_pdf, 10000)

c = ROOT.TCanvas("c", "c", 700, 500)
c.SetLeftMargin(0.15)
h_final.SetStats(0)
h_final.GetXaxis().SetTitle("Fake Reconstructed Z [cm]")
h_final.GetYaxis().SetTitle("Fake Track Count")
h_final.Draw("HIST")

c.Draw()